# Phase 5 - All Structures

Runs the full pipeline (regimes -> fair value -> signals) for every structure using `pipeline.py`:
- **Calendar (c1-c2)** and **Butterfly (c1-2c2+c3)** for CL, HO, LCO, LGO, and the WTI-Brent spread curve.
- **Cracks:** heating (HO*42 - CL) on the CL regime; gasoil (LGO/7.45 - LCO) on the Brent regime.

Regimes are computed **once per instrument** (national inventory + that instrument's vol & curve) and shared by its structures. Output: one combined signal book `cache/all_signals.parquet`.

In [1]:
import importlib, pipeline as P
importlib.reload(P)
import pandas as pd, numpy as np
from pathlib import Path
CACHE=Path("cache")
INSTRUMENTS=["CL","HO","LCO","LGO","wtcl_lco_outrights"]
LABEL={"CL":"WTI","HO":"HeatingOil","LCO":"Brent","LGO":"Gasoil","wtcl_lco_outrights":"WTI-Brent"}

panels={}; regs={}
for inst in INSTRUMENTS:
    f=P.daily_panel(inst)
    panel=f.join(P.fetch_fundamentals(f.index))
    regs[inst]=P.score_regimes(panel); panels[inst]=f
    print(f"{inst:20s} rows={len(regs[inst]):4d} regimes={regs[inst]['regime_eff'].nunique()}")

CL                   rows=1408 regimes=15


HO                   rows=1369 regimes=15


LCO                  rows=1319 regimes=12


LGO                  rows=1310 regimes=15


wtcl_lco_outrights   rows=   0 regimes=0


C:\Users\yash.yeole\AppData\Roaming\Python\Python311\site-packages\pandas\core\arraylike.py:402: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


## Calendar + Butterfly for every instrument

In [2]:
books=[]
for inst in INSTRUMENTS:
    for struct,target in [("c1-c2","cal"),("butterfly","fly")]:
        sig=P.run_structure(regs[inst].copy(), target, LABEL[inst], struct)
        if sig is not None:
            books.append(sig);
            te=sig[sig.split=='test']
            print(f"{LABEL[inst]:10s} {struct:9s} model={sig['_chosen'].iloc[0]:5s} "
                  f"test_OOD={int(te.ood.sum()):2d}/{len(te)} "
                  f"active_eps={sig['episode'].nunique()}")

WTI        c1-c2     model=enet  test_OOD=45/69 active_eps=64


WTI        butterfly model=ridge test_OOD=27/69 active_eps=79


HeatingOil c1-c2     model=enet  test_OOD= 8/68 active_eps=42


HeatingOil butterfly model=ridge test_OOD= 8/68 active_eps=63


Brent      c1-c2     model=enet  test_OOD=59/70 active_eps=33


Brent      butterfly model=enet  test_OOD=28/70 active_eps=46


Gasoil     c1-c2     model=enet  test_OOD=30/68 active_eps=42


Gasoil     butterfly model=enet  test_OOD=20/68 active_eps=40


## Cracks (cross-instrument, on the crude regime)

In [3]:
def add_crack(crude, prod, name, conv):
    r=regs[crude].copy()
    p1=panels[prod]["c1"].reindex(r.index)
    r[name]=p1*conv - r["c1"]
    return P.run_structure(r, name, LABEL[crude]+"-"+LABEL[prod], name)

heat=add_crack("CL","HO","crack_heating",42.0)       # $/gal -> $/bbl
gas =add_crack("LCO","LGO","crack_gasoil",1/7.45)    # $/tonne -> $/bbl
for sig,nm in [(heat,"heating crack"),(gas,"gasoil crack")]:
    if sig is not None:
        books.append(sig); te=sig[sig.split=='test']
        print(f"{nm:14s} model={sig['_chosen'].iloc[0]:5s} test_OOD={int(te.ood.sum())}/{len(te)}")

heating crack  model=enet  test_OOD=8/67
gasoil crack   model=enet  test_OOD=43/68


## Combine & save the signal book

In [4]:
cols=["instrument","structure","actual","fair_value","residual","z","direction",
      "state","ood","episode","confidence","regime_eff","regime_level","near_boundary",
      "regime_n","resid_std_train","split"]
allsig=pd.concat([b.assign(date=b.index)[["date"]+cols] for b in books], ignore_index=True)
allsig=allsig.set_index("date").sort_index()
allsig.to_parquet(CACHE/"all_signals.parquet")
print("saved all_signals.parquet | rows:", len(allsig),
      "| structures:", allsig.groupby(['instrument','structure']).ngroups)
allsig.groupby(["instrument","structure"]).size().rename("rows")

saved all_signals.parquet | rows: 13497 | structures: 10


instrument      structure    
Brent           butterfly        1319
                c1-c2            1319
Brent-Gasoil    crack_gasoil     1310
Gasoil          butterfly        1310
                c1-c2            1310
HeatingOil      butterfly        1369
                c1-c2            1369
WTI             butterfly        1408
                c1-c2            1408
WTI-HeatingOil  crack_heating    1375
Name: rows, dtype: int64

## Sanity: latest cross-sectional opportunity board

In [5]:
latest=allsig.index.max()
cur=allsig.loc[[latest]] if latest in allsig.index else allsig.tail(20)
board=cur[cur.state.isin(["ENTRY","HOLD"])].assign(absz=lambda d:d.z.abs())
print("as of", latest.date())
board.sort_values("absz",ascending=False)[["instrument","structure","regime_eff",
    "actual","fair_value","z","direction","confidence"]].round(2)

as of 2026-05-22


,instrument,structure,regime_eff,actual,fair_value,z,direction,confidence
date,,,,,,,,
2026-05-22,Brent,c1-c2,M|H|Back,3.37,1.71,2.45,SHORT,HIGH
2026-05-22,Brent,butterfly,M|H|Back,-0.30,0.33,-1.70,LONG,HIGH
